# Stage 1 ML Experimentation & Model Development Notebook
**Project**: Personalized Precision Medicine for Oncology Treatment Optimization  
**Task**: Supervised Multiclass Classification (`toxicity_risk`: `Low`, `Moderate`, `High`)

This notebook demonstrates the end-to-end reproducible machine learning pipeline, including patient-level train/test splitting, domain feature engineering, preprocessor fitting, cross-validation model benchmarking, hyperparameter tuning, locked test set evaluation, feature importances, and sample inference.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src modules are imported
sys.path.insert(0, os.path.abspath('../src'))

from data_loader import load_master_dataset, prepare_features_and_target, NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from feature_engineering import FeatureEngineer
from preprocessing import PreprocessingArtifactManager
from utils import patient_level_split, evaluate_multiclass_predictions, plot_confusion_matrix_figure
from predict import predict_patient_toxicity

print('Imports successfully loaded.')

## 1. Dataset Loading & Validation

In [ ]:
raw_df = load_master_dataset()
print('Dataset Shape:', raw_df.shape)
print('Target Distribution:')
print(raw_df['toxicity_risk'].value_counts())

## 2. Patient-Level Stratified Split (80% Train / 20% Locked Test)

In [ ]:
train_df, test_df = patient_level_split(raw_df, group_col='patient_id', target_col='toxicity_risk', test_size=0.20, random_state=42)
train_pids = set(train_df['patient_id'])
test_pids = set(test_df['patient_id'])
overlap = train_pids.intersection(test_pids)
print(f'Train Encounters: {len(train_df)} ({len(train_pids)} patients)')
print(f'Test Encounters: {len(test_df)} ({len(test_pids)} patients)')
print(f'Patient Overlap Count: {len(overlap)} (MUST BE 0)')

## 3. Feature Engineering & Preprocessing

In [ ]:
X_train_raw, y_train, meta_train = prepare_features_and_target(train_df)
X_test_raw, y_test, meta_test = prepare_features_and_target(test_df)

fe = FeatureEngineer(include_engineered=True)
X_train_fe = fe.transform(X_train_raw)
X_test_fe = fe.transform(X_test_raw)

num_cols = list(NUMERICAL_FEATURES) + [
    'blood_pressure_ratio', 'pulse_pressure', 'comorbidity_age_interaction',
    'tumor_biomarker_index', 'hematologic_risk_flag', 'prior_toxicity_risk_flag'
]
cat_cols = list(CATEGORICAL_FEATURES)

pm = PreprocessingArtifactManager()
X_train_proc = pm.fit_transform(X_train_fe, num_cols, cat_cols)
X_test_proc = pm.transform(X_test_fe)

print(f'Processed Training Matrix Shape: {X_train_proc.shape}')
print(f'Processed Test Matrix Shape: {X_test_proc.shape}')

## 4. Model Benchmarking & Final Test Evaluation

In [ ]:
import joblib
best_model = joblib.load('../models/best_model/model.joblib')
y_test_pred = best_model.predict(X_test_proc)
metrics = evaluate_multiclass_predictions(y_test.values, y_test_pred)
print('Final Locked Test Set Performance Metrics:')
print(f"Accuracy:        {metrics['accuracy']:.4f}")
print(f"Macro F1:        {metrics['macro_f1']:.4f}")
print(f"Weighted F1:     {metrics['weighted_f1']:.4f}")
print(f"High-Risk Rec:   {metrics['high_risk_recall']:.4f}")

## 5. Feature Importance Visualization

In [ ]:
df_imp = pd.read_csv('../results/feature_importance.csv')
plt.figure(figsize=(8, 5))
sns.barplot(data=df_imp.head(10), x='importance', y='feature', hue='feature', legend=False, palette='viridis')
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 6. Sample Single-Patient Inference

In [ ]:
sample_patient = {
    'age': 62.0, 'sex': 'Female', 'cancer_type': 'NSCLC', 'cancer_stage': 'Stage III',
    'smoking_history': 'Former', 'mutation_primary': 'EGFR', 'mutation_secondary': 'TP53',
    'mutation_burden': 6.8, 'gene_expression_score': 49.8, 'ctdna_level': 1.4,
    'tumor_marker_level': 24.7, 'inflammation_marker': 4.2, 'biomarker_trend': 'Stable',
    'heart_rate': 79.0, 'systolic_bp': 128.0, 'diastolic_bp': 80.0, 'oxygen_saturation': 96.0,
    'hemoglobin': 12.5, 'white_blood_cell_count': 7.41, 'platelet_count': 231.0,
    'creatinine_level': 1.0, 'liver_function_marker': 17.3, 'treatment_type': 'Targeted Therapy',
    'drug_name': 'Erlotinib', 'drug_dose': 150.0, 'treatment_cycle': 4,
    'previous_treatment_count': 1, 'previous_adverse_event': False, 'previous_toxicity_grade': 0.0,
    'comorbidity_count': 1.0
}
res = predict_patient_toxicity(sample_patient)
print(json.dumps(res, indent=2))